# Rotation angle 비교: scipy `as_rotvec()` vs eigenvalue

`python/verifytool/verifypositioner.py`의 `_compute_delta()`에서는 상대 회전 `rel`에 대해

```python
angle_deg = degrees(norm(rel.as_rotvec()))
```

으로 전체 회전각을 구합니다. 이 노트북은 더미 quaternion/position 데이터를 만들어서 같은 angle을 eigenvalue 기반 방식과 비교합니다.

비교 방식:

- **scipy rotvec**: `||Rotation.as_rotvec()||`
- **eigen phase**: 회전행렬의 고유값 `{1, exp(+i theta), exp(-i theta)}`에서 phase의 최대 절댓값
- **trace formula**: `acos((trace(R) - 1) / 2)`; eigenvalue 방식의 sanity check 용도

참고: scipy quaternion 순서는 `[x, y, z, w]`입니다.

In [ ]:
import csv
import math
from datetime import datetime
from pathlib import Path
import numpy as np
from scipy.spatial.transform import Rotation

np.set_printoptions(precision=8, suppress=True)
rng = np.random.default_rng(20260615)

## Helper functions

`compute_delta_like()`는 현재 `_compute_delta()`에서 쓰는 상대 회전 계산 흐름을 최대한 비슷하게 흉내냅니다.

In [ ]:
def angle_deg_scipy_rotvec(rel: Rotation) -> float:
    return float(np.degrees(np.linalg.norm(rel.as_rotvec())))


def angle_deg_eigen_phase(R: np.ndarray) -> float:
    """Angle from eigenvalues of a 3D rotation matrix.

    A proper 3D rotation matrix has eigenvalues 1, exp(+i theta), exp(-i theta).
    Taking max(abs(angle(lambda))) returns theta in [0, pi].
    """
    evals = np.linalg.eigvals(R)
    theta = np.max(np.abs(np.angle(evals)))
    return float(np.degrees(theta))


def angle_deg_trace(R: np.ndarray) -> float:
    cos_theta = (np.trace(R) - 1.0) / 2.0
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_theta)))


def axis_eigen_lambda_one(R: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    """Axis from the eigenvector whose eigenvalue is closest to 1.

    The axis is undefined for identity or near-identity rotations, so this returns NaNs
    when every direction is effectively an axis.
    """
    evals, evecs = np.linalg.eig(R)
    best = int(np.argmin(np.abs(evals - 1.0)))
    if abs(evals[best] - 1.0) > 1e-5:
        return np.array([np.nan, np.nan, np.nan])

    axis = evecs[:, best].real
    norm = np.linalg.norm(axis)
    if norm < eps:
        return np.array([np.nan, np.nan, np.nan])
    return axis / norm


def compute_delta_like(start_pos, start_quat, stop_pos, stop_quat):
    start_rot = Rotation.from_quat(start_quat)
    stop_rot = Rotation.from_quat(stop_quat)
    rel = start_rot.inv() * stop_rot
    R = rel.as_matrix()
    yaw, pitch, roll = rel.as_euler("ZYX", degrees=True)
    dp_mm = (np.asarray(stop_pos) - np.asarray(start_pos)) * 1000.0

    return {
        "dx_mm": float(dp_mm[0]),
        "dy_mm": float(dp_mm[1]),
        "dz_mm": float(dp_mm[2]),
        "droll_deg": float(roll),
        "dpitch_deg": float(pitch),
        "dyaw_deg": float(yaw),
        "angle_scipy_deg": angle_deg_scipy_rotvec(rel),
        "angle_eigen_phase_deg": angle_deg_eigen_phase(R),
        "angle_trace_deg": angle_deg_trace(R),
        "axis_eigen": axis_eigen_lambda_one(R),
        "eigenvalues": np.linalg.eigvals(R),
    }


def unit_random_axis():
    v = rng.normal(size=3)
    return v / np.linalg.norm(v)


def fmt_row(values, widths):
    return " | ".join(str(v).rjust(w) for v, w in zip(values, widths))


def print_table(headers, rows):
    widths = [len(h) for h in headers]
    for row in rows:
        widths = [max(w, len(str(v))) for w, v in zip(widths, row)]
    print(fmt_row(headers, widths))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(fmt_row(row, widths))

## Deterministic dummy cases

작은 각도, 일반적인 각도, 180도 근처를 일부러 섞습니다. axis는 부호가 뒤집혀도 같은 회전축이므로 `abs(dot(true_axis, eigen_axis))`로 비교합니다.

In [ ]:
case_angles_deg = [0.0, 1e-6, 0.001, 0.1, 1.0, 30.0, 90.0, 179.999, 180.0]
rows = []

for target_angle_deg in case_angles_deg:
    start_rot = Rotation.random(random_state=rng)
    true_axis = unit_random_axis()
    rel_rot = Rotation.from_rotvec(np.radians(target_angle_deg) * true_axis)
    stop_rot = start_rot * rel_rot

    start_pos = rng.uniform(-1.0, 1.0, size=3)
    stop_pos = start_pos + rng.normal(scale=0.002, size=3)
    result = compute_delta_like(start_pos, start_rot.as_quat(), stop_pos, stop_rot.as_quat())

    eigen_axis = result["axis_eigen"]
    axis_abs_dot = np.nan
    if np.all(np.isfinite(eigen_axis)) and target_angle_deg > 1e-5:
        axis_abs_dot = abs(float(np.dot(true_axis, eigen_axis)))

    rows.append([
        f"{target_angle_deg:.6f}",
        f"{result['angle_scipy_deg']:.12f}",
        f"{result['angle_eigen_phase_deg']:.12f}",
        f"{result['angle_trace_deg']:.12f}",
        f"{abs(result['angle_scipy_deg'] - result['angle_eigen_phase_deg']):.3e}",
        "nan" if np.isnan(axis_abs_dot) else f"{axis_abs_dot:.12f}",
    ])

print_table(
    ["target_deg", "scipy_deg", "eigen_phase_deg", "trace_deg", "|scipy-eigen|", "axis_abs_dot"],
    rows,
)

## Monte Carlo comparison

랜덤 상대 회전 10,000개에 대해 두 방식의 차이를 봅니다.

In [ ]:
n = 10_000
errors_eigen = []
errors_trace = []

for _ in range(n):
    rel = Rotation.random(random_state=rng)
    R = rel.as_matrix()
    a_scipy = angle_deg_scipy_rotvec(rel)
    errors_eigen.append(abs(a_scipy - angle_deg_eigen_phase(R)))
    errors_trace.append(abs(a_scipy - angle_deg_trace(R)))

errors_eigen = np.asarray(errors_eigen)
errors_trace = np.asarray(errors_trace)

print("eigen phase error deg")
print("  max :", errors_eigen.max())
print("  mean:", errors_eigen.mean())
print("  p99 :", np.percentile(errors_eigen, 99))
print()
print("trace formula error deg")
print("  max :", errors_trace.max())
print("  mean:", errors_trace.mean())
print("  p99 :", np.percentile(errors_trace, 99))

## 한 샘플 자세히 보기

현재 코드와 비슷하게 position delta, Euler delta, angle, axis, eigenvalues를 한 번에 확인합니다.

In [ ]:
start_pos = np.array([0.10, -0.20, 0.30])
stop_pos = np.array([0.1015, -0.198, 0.299])

start_rot = Rotation.from_euler("ZYX", [15.0, -8.0, 3.0], degrees=True)
rel_rot = Rotation.from_rotvec(np.radians(12.5) * np.array([0.2, -0.5, 0.84]) / np.linalg.norm([0.2, -0.5, 0.84]))
stop_rot = start_rot * rel_rot

sample = compute_delta_like(start_pos, start_rot.as_quat(), stop_pos, stop_rot.as_quat())

for key, value in sample.items():
    if isinstance(value, np.ndarray):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {value}")

## 실제 positioner CSV 검증

`positioner_20260611_161133.csv` 같은 요약 CSV를 읽어서 저장된 `angle_deg`와 eigenvalue 기반 angle을 비교합니다.

같은 prefix의 trajectory 파일, 예를 들어 `positioner_20260611_161133_rb1_traj.csv`, `positioner_20260611_161133_rb2_traj.csv`가 있으면 trajectory의 첫 pose와 마지막 pose로 `_compute_delta()` 스타일 값을 다시 계산해서 저장된 요약값과 비교합니다.

In [ ]:
# 다른 파일을 검증하려면 이 파일명/경로만 바꾸면 됩니다.
POSITIONER_CSV_INPUT = "positioner_verification_data/positioner_20260611_161133.csv"


def resolve_positioner_csv(path_like):
    path = Path(path_like)
    candidates = [path, Path("..") / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"CSV not found. Tried: {candidates}")


POSITIONER_CSV = resolve_positioner_csv(POSITIONER_CSV_INPUT)


def to_float(row, key, default=np.nan):
    value = row.get(key, "")
    if value is None or value == "":
        return default
    return float(value)


def read_csv_rows(path: Path):
    with path.open("r", newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def eigenvalues_from_summary_row(row):
    return np.array([
        complex(to_float(row, "eval1_re"), to_float(row, "eval1_im", 0.0)),
        complex(to_float(row, "eval2_re"), to_float(row, "eval2_im", 0.0)),
        complex(to_float(row, "eval3_re"), to_float(row, "eval3_im", 0.0)),
    ])


def angle_deg_eigenvalues(evals):
    return float(np.degrees(np.max(np.abs(np.angle(evals)))))


def summarize_saved_csv(path: Path):
    rows = read_csv_rows(path)
    out = []
    for row in rows:
        evals = eigenvalues_from_summary_row(row)
        saved = to_float(row, "angle_deg")
        eig_angle = angle_deg_eigenvalues(evals)
        out.append([
            row.get("rb", row.get("rb_id", "?")),
            f"{saved:.12f}",
            f"{eig_angle:.12f}",
            f"{abs(saved - eig_angle):.3e}",
            f"{evals[0]:.8f}",
            f"{evals[1]:.8f}",
            f"{evals[2]:.8f}",
        ])
    print(f"summary csv: {path}")
    print_table(
        ["rb", "saved_angle_deg", "eigen_from_saved_evals", "abs_diff", "eval1", "eval2", "eval3"],
        out,
    )
    return rows


summary_rows = summarize_saved_csv(POSITIONER_CSV)

## Trajectory 파일이 있을 때 재계산

요약 CSV만 있으면 저장된 eigenvalue와 저장된 angle의 일관성만 볼 수 있습니다. trajectory 파일이 같이 있으면 start/end quaternion으로 scipy `as_rotvec()` angle까지 다시 계산할 수 있습니다.

In [ ]:
def traj_path_for_summary(summary_path: Path, rb):
    return summary_path.with_name(f"{summary_path.stem}_rb{rb}_traj.csv")


def pose_from_traj_row(row):
    pos = np.array([to_float(row, "x_m"), to_float(row, "y_m"), to_float(row, "z_m")])
    quat = [to_float(row, "qx"), to_float(row, "qy"), to_float(row, "qz"), to_float(row, "qw")]
    return pos, quat


def recompute_from_traj(summary_path: Path, summary_rows):
    rows = []
    for summary in summary_rows:
        rb = summary.get("rb", summary.get("rb_id"))
        traj_path = traj_path_for_summary(summary_path, rb)
        if not traj_path.exists():
            rows.append([rb, "missing", "", "", "", "", str(traj_path)])
            continue

        traj_rows = read_csv_rows(traj_path)
        if len(traj_rows) < 2:
            rows.append([rb, "too few rows", "", "", "", "", str(traj_path)])
            continue

        start_pos, start_quat = pose_from_traj_row(traj_rows[0])
        stop_pos, stop_quat = pose_from_traj_row(traj_rows[-1])
        recomputed = compute_delta_like(start_pos, start_quat, stop_pos, stop_quat)
        saved_angle = to_float(summary, "angle_deg")
        saved_dx = to_float(summary, "dx_mm")
        saved_dy = to_float(summary, "dy_mm")
        saved_dz = to_float(summary, "dz_mm")
        saved_dp = np.array([saved_dx, saved_dy, saved_dz])
        recomputed_dp = np.array([recomputed["dx_mm"], recomputed["dy_mm"], recomputed["dz_mm"]])

        rows.append([
            rb,
            len(traj_rows),
            f"{saved_angle:.12f}",
            f"{recomputed['angle_scipy_deg']:.12f}",
            f"{recomputed['angle_eigen_phase_deg']:.12f}",
            f"{abs(saved_angle - recomputed['angle_scipy_deg']):.3e}",
            f"{np.linalg.norm(saved_dp - recomputed_dp):.3e}",
        ])

    print_table(
        ["rb", "traj_rows", "saved_angle", "traj_scipy", "traj_eigen", "angle_abs_diff", "dp_norm_diff_mm"],
        rows,
    )


recompute_from_traj(POSITIONER_CSV, summary_rows)

## 폴더 전체 일괄 검증 및 로그 저장

`positioner_verification_data/` 안에서 `_traj.csv`로 끝나지 않는 모든 CSV를 검증합니다. 요약 CSV의 저장 angle과 저장 eigenvalue phase가 `1e-9 deg` 이내로 일치하면 PASS입니다. trajectory 첫/끝 pose와의 차이는 start/stop snapshot이 trajectory 수집 구간 바깥에서 잡히는 현재 저장 구조를 고려해 참고 지표로만 기록합니다.

In [ ]:
VERIFICATION_DIR_INPUT = Path("positioner_verification_data")
ANGLE_TOL_DEG = 1e-9
EIGEN_MODULUS_TOL = 1e-9


def resolve_existing_dir(path: Path):
    for candidate in [path, Path("..") / path]:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(f"Directory not found: {path}")


def validate_positioner_folder(folder: Path):
    folder = resolve_existing_dir(folder)
    summary_paths = sorted(p for p in folder.glob("*.csv") if not p.name.endswith("_traj.csv"))
    summary_paths = [p for p in summary_paths if p.name != "positioner_verification_results.csv"]
    detail_rows = []

    for summary_path in summary_paths:
        try:
            summaries = read_csv_rows(summary_path)
        except Exception as exc:
            detail_rows.append({"summary_file": summary_path.name, "status": "ERROR", "message": str(exc)})
            continue

        for summary in summaries:
            rb = summary.get("rb", summary.get("rb_id", "?"))
            row = {"summary_file": summary_path.name, "rb": rb, "status": "ERROR", "message": ""}
            try:
                evals = eigenvalues_from_summary_row(summary)
                saved_angle = to_float(summary, "angle_deg")
                eigen_angle = angle_deg_eigenvalues(evals)
                angle_diff = abs(saved_angle - eigen_angle)
                modulus_error = float(np.max(np.abs(np.abs(evals) - 1.0)))
                internal_ok = angle_diff <= ANGLE_TOL_DEG and modulus_error <= EIGEN_MODULUS_TOL

                row.update({
                    "status": "PASS" if internal_ok else "FAIL",
                    "saved_angle_deg": saved_angle,
                    "eigen_angle_deg": eigen_angle,
                    "angle_abs_diff_deg": angle_diff,
                    "max_eigen_modulus_error": modulus_error,
                    "trajectory_file": "",
                    "trajectory_rows": "",
                    "traj_scipy_angle_deg": "",
                    "traj_eigen_angle_deg": "",
                    "saved_vs_traj_angle_diff_deg": "",
                    "saved_vs_traj_dp_diff_mm": "",
                })

                traj_path = traj_path_for_summary(summary_path, rb)
                row["trajectory_file"] = traj_path.name
                if traj_path.exists():
                    traj_rows = read_csv_rows(traj_path)
                    row["trajectory_rows"] = len(traj_rows)
                    if len(traj_rows) >= 2:
                        start_pos, start_quat = pose_from_traj_row(traj_rows[0])
                        stop_pos, stop_quat = pose_from_traj_row(traj_rows[-1])
                        recomputed = compute_delta_like(start_pos, start_quat, stop_pos, stop_quat)
                        saved_dp = np.array([to_float(summary, k) for k in ("dx_mm", "dy_mm", "dz_mm")])
                        traj_dp = np.array([recomputed[k] for k in ("dx_mm", "dy_mm", "dz_mm")])
                        row.update({
                            "traj_scipy_angle_deg": recomputed["angle_scipy_deg"],
                            "traj_eigen_angle_deg": recomputed["angle_eigen_phase_deg"],
                            "saved_vs_traj_angle_diff_deg": abs(saved_angle - recomputed["angle_scipy_deg"]),
                            "saved_vs_traj_dp_diff_mm": float(np.linalg.norm(saved_dp - traj_dp)),
                        })
                else:
                    row["message"] = "trajectory file missing"
            except Exception as exc:
                row["status"] = "ERROR"
                row["message"] = str(exc)
            detail_rows.append(row)

    result_path = folder / "positioner_verification_results.csv"
    log_path = folder / "positioner_verification.log"
    fieldnames = [
        "summary_file", "rb", "status", "saved_angle_deg", "eigen_angle_deg",
        "angle_abs_diff_deg", "max_eigen_modulus_error", "trajectory_file",
        "trajectory_rows", "traj_scipy_angle_deg", "traj_eigen_angle_deg",
        "saved_vs_traj_angle_diff_deg", "saved_vs_traj_dp_diff_mm", "message",
    ]
    with result_path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(detail_rows)

    counts = {key: sum(r.get("status") == key for r in detail_rows) for key in ("PASS", "FAIL", "ERROR")}
    valid_diffs = [r["angle_abs_diff_deg"] for r in detail_rows if isinstance(r.get("angle_abs_diff_deg"), (int, float))]
    traj_diffs = [r["saved_vs_traj_angle_diff_deg"] for r in detail_rows if isinstance(r.get("saved_vs_traj_angle_diff_deg"), (int, float))]
    lines = [
        "Positioner rotation verification",
        f"generated_at: {datetime.now().isoformat(timespec='seconds')}",
        f"folder: {folder.resolve()}",
        f"summary_files: {len(summary_paths)}",
        f"rows: {len(detail_rows)}",
        f"PASS: {counts['PASS']}",
        f"FAIL: {counts['FAIL']}",
        f"ERROR: {counts['ERROR']}",
        f"angle_tolerance_deg: {ANGLE_TOL_DEG}",
        f"eigen_modulus_tolerance: {EIGEN_MODULUS_TOL}",
        f"max_saved_vs_eigen_angle_diff_deg: {max(valid_diffs) if valid_diffs else 'n/a'}",
        f"max_saved_vs_traj_angle_diff_deg: {max(traj_diffs) if traj_diffs else 'n/a'}",
        "",
        "FAIL/ERROR rows:",
    ]
    bad_rows = [r for r in detail_rows if r.get("status") != "PASS"]
    lines.extend(
        f"- {r.get('summary_file')} RB{r.get('rb')}: {r.get('status')} {r.get('message', '')}"
        for r in bad_rows
    )
    if not bad_rows:
        lines.append("- none")
    log_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    print("\n".join(lines))
    print(f"detail csv: {result_path}")
    print(f"text log  : {log_path}")
    return detail_rows


batch_results = validate_positioner_folder(VERIFICATION_DIR_INPUT)

## Notes

- `as_rotvec()`와 eigenvalue phase는 정상적인 3D rotation matrix에서는 같은 angle을 줍니다.
- identity 또는 아주 작은 회전에서는 회전축이 수학적으로 거의/완전히 정의되지 않습니다. angle 비교는 가능하지만 axis 비교는 조심해야 합니다.
- 180도 근처에서는 고유값 `-1, -1, 1` 때문에 고유공간이 민감할 수 있습니다. angle은 잘 나오지만 axis는 행렬 노이즈가 있으면 흔들릴 수 있습니다.
- 실제 `_compute_delta()`의 axis 추출 코드는 `np.argmax(real_mask & condition)` 형태라 조건을 만족하는 eigenvalue가 없을 때 index 0을 고를 수 있습니다. 더 방어적으로 쓰려면 `argmin(abs(evals - 1))` 방식이 안전합니다.